# Data Pipeline

This notebook documents how nhanes_clean.csv was produced from the raw NHANES 2021-23 survey files. 
It is provided for transparency and does not need to be re-run, saved/nhanes_clean.csv is already 
included in the repo and is the starting point for model.ipynb.

To reproduce from raw: download the 2021-23 NHANES XPT files from 
https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?Cycle=2021-2023, merge on SEQN, 
and save as NHANES_data.csv in the repo root before running.

In [21]:
# Load libraries and packages
import numpy as np
import pandas as pd
import os
import urllib.request
from pathlib import Path

In [22]:
# Section 0: Data Merge
"""
Download the following XPT files manually from the CDC NHANES 2021-23 portal:
https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?Cycle=2021-2023

Files needed (under Questionnaire, Examination, and Laboratory Data):
   DEMO_L, BMX_L, BPXO_L, BPQ_L, DIQ_L, MCQ_L, KIQ_U_L,
   HIQ_L, HUQ_L, SLQ_L, SMQ_L, ALQ_L, PAQ_L, OCQ_L, WHQ_L, DBQ_L

Place all downloaded .XPT files into a folder called raw_nhanes/ in the repo root,
then run this cell.
"""

#Below commented out.
"""
raw_dir = Path("raw_nhanes")
if not raw_dir.exists():
    raise FileNotFoundError("raw_nhanes/ folder not found. See instructions above.")

frames = []
for xpt_file in sorted(raw_dir.glob("*.XPT")):
    df = pd.read_sas(xpt_file, format="xport", encoding="utf-8")
    if "SEQN" not in df.columns:
        continue
    if df["SEQN"].duplicated().any():
        df = df.drop_duplicates(subset="SEQN", keep="first")
    frames.append(df.set_index("SEQN"))
    print(f"  Loaded {xpt_file.name}: {df.shape[0]:,} rows")

data = frames[0].join(frames[1:], how="outer").reset_index()
print(f"\nMerged: {data.shape[0]:,} participants x {data.shape[1]:,} columns")

data.to_csv("NHANES_data.csv", index=False)
print("Saved to NHANES_data.csv")
"""

'\nraw_dir = Path("raw_nhanes")\nif not raw_dir.exists():\n    raise FileNotFoundError("raw_nhanes/ folder not found. See instructions above.")\n\nframes = []\nfor xpt_file in sorted(raw_dir.glob("*.XPT")):\n    df = pd.read_sas(xpt_file, format="xport", encoding="utf-8")\n    if "SEQN" not in df.columns:\n        continue\n    if df["SEQN"].duplicated().any():\n        df = df.drop_duplicates(subset="SEQN", keep="first")\n    frames.append(df.set_index("SEQN"))\n    print(f"  Loaded {xpt_file.name}: {df.shape[0]:,} rows")\n\ndata = frames[0].join(frames[1:], how="outer").reset_index()\nprint(f"\nMerged: {data.shape[0]:,} participants x {data.shape[1]:,} columns")\n\ndata.to_csv("NHANES_data.csv", index=False)\nprint("Saved to NHANES_data.csv")\n'

In [23]:
# Raw NHANES extract, place the NHANES_data.csv in the repo root before running.
# Download from: https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?Cycle=2021-2023 merging files on SEQN (user id).
# (or use the pre-processed nhanes_clean.csv in saved/ and skip to model.ipynb)
# path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "NHANES_data.csv")
data = pd.read_csv('../NHANES_data.csv') # local path
# data = pd.read_csv(path)

/var/folders/h6/7yp_ymqd6mq5hrczq9ztfc580000gn/T/ipykernel_64630/1140571439.py:5: DtypeWarning: Columns (2583,2584,2585,2586) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('../NHANES_data.csv') # local path


In [24]:
# Identifying all my relevant inputs and outputs
our_cols = [
    # demographics
    'RIDAGEYR','RIAGENDR','RIDRETH3','BMXBMI','BMXWAIST',
    'SMQ020','SMQ040','ALQ121','PAQ706','HIQ011',
    'SLQ300','SLQ310','HUQ010','HUQ090','OCQ180','WHQ070','DBQ095Z',
    'BPXOSY1','BPXOSY2','BPXOSY3','BPXODI1','BPXODI2','BPXODI3',
    # diseases
    'DIQ010','DIQ160','BPQ020','BPQ080','MCQ160A','MCQ160B',
    'MCQ160C','MCQ160E','MCQ160F','MCQ160L','MCQ160M','MCQ160P',
    'MCQ220','KIQ022','MCQ010'
]
data = data[our_cols]

In [25]:
# Preprocessing
"""
After looking at our data, quite a bit needs filtering, we start with:
(1) Find averages of blood pressure readings
(2) Recode diseases columns to categorical and generally 1 (yes) and 0 (no) from initial 1,2,7,9 scale.
(3) Drop certain categorical values such as 7 or 9 which often means they didn't or refused to answer
(4) Recode race and salt intake to be one hot encoded
"""

# Using the average of three readings (if have <3, ignores NaNs automatically)
data['SBP'] = data[['BPXOSY1','BPXOSY2','BPXOSY3']].mean(axis=1)
data['DBP'] = data[['BPXODI1','BPXODI2','BPXODI3']].mean(axis=1)

# Recoding disease columns and some demogrpahics as Yes=1 and No=0, annotated with question as they all follow 1,2,7,9 format
recode_cols = [
    'DIQ160',    # "Has a doctor ever told you that you have pre-diabetes?"
    'BPQ020',    # "Has a doctor ever told you that you have hypertension?"
    'BPQ080',    # "Has a doctor ever told you your cholesterol level was high?"
    'MCQ160A',   # "Has a doctor ever told you that you have arthritis?"
    'MCQ160B',   # "Has a doctor ever told you that you have congestive heart failure?"
    'MCQ160C',   # "Has a doctor ever told you that you have coronary heart disease?"
    'MCQ160E',   # "Has a doctor ever told you that you have had a heart attack?"
    'MCQ160F',   # "Has a doctor ever told you that you have had a stroke?"
    'MCQ160L',   # "Has a doctor ever told you that you have a liver condition?"
    'MCQ160M',   # "Has a doctor ever told you that you have a thyroid problem?"
    'MCQ160P',   # "Has a doctor ever told you that you have COPD or emphysema?"
    'MCQ220',    # "Has a doctor ever told you that you have cancer or malignancy?"
    'KIQ022',    # "Has a doctor ever told you that you have kidney disease?"
    'MCQ010',    # "Has a doctor ever told you that you have asthma?"
    'SMQ020',    # "Have you smoked at least 100 cigarettes in your life?"
    'HIQ011',    # "Are you covered by health insurance?"
    'HUQ090',    # "In the past year, have you seen a mental health professional?"
    'WHQ070',    # "In the past year, have you tried to lose weight?"
]

for col in recode_cols:
    data[col] = data[col].replace({
        1: 1,    # Yes stays as 1
        2: 0,    # No goes from 2 to 0
        7: np.nan,  # Refused becomes a missing (NaN)
        9: np.nan,  # Don't know also becomes missing (NaN)
    })

# SMQ040 (How often do you smoke): we make it 0 not at all, 1 sometimes, to 2 most of the times
# 1=Every day becomes 2 (most), 2=Some days becomes 1 (moderate) and 3=Not at all becomes 0 (none)
data['SMQ040'] = data['SMQ040'].replace({1: 2, 2: 1, 3: 0, 7: np.nan, 9: np.nan})

# RIAGENDR: Sex, recode to 0=Male, 1=Female as +ive SHAP value = being female increases predicted risk
data['RIAGENDR'] = data['RIAGENDR'].replace({1:0, 2:1})

# RIDRETH3: Race/ethnicity — one-hot encode, Non-Hispanic White (3) as reference
data['RIDRETH3'] = data['RIDRETH3'].astype('Int64') #convert to Int64 
data = pd.get_dummies(data, columns=['RIDRETH3'], prefix='RACE', dummy_na=False)
data = data.drop(columns=['RACE_3'], errors='ignore')  # Non-Hispanic White is base ref
"""
Codes are as follows:
1 = Mexican American
2 = Other Hispanic  
3 = Non-Hispanic White - make this the base case as its the largest demographic in our sample
4 = Non-Hispanic Black
5 = Other Race / Multi-racial
6 = Non-Hispanic Asian
7 = Other
"""

# HUQ010: "How would you rate your general health?" 1=Excellent Poor=5 so just reverse for intuitive SHAP
data['HUQ010'] = data['HUQ010'].replace({7:np.nan, 9:np.nan})
data['HUQ010'] = data['HUQ010'].replace({1:5, 2:4, 3:3, 4:2, 5:1})

# DBQ095Z: "What type of salt do you use?" Categorical, so one-hot encode rather than treat as ordinal
data['DBQ095Z'] = data['DBQ095Z'].replace({7:np.nan, 9:np.nan, 99:np.nan}) # convert to Int6
data['DBQ095Z'] = data['DBQ095Z'].astype('Int64')
data = pd.get_dummies(data, columns=['DBQ095Z'], prefix='SALT', dummy_na=False)
# creates SALT_1 to SALT_6 binary columns and below we drop one to avoid dummy variable trap
data = data.drop(columns=['SALT_1'], errors='ignore')  # "never adds salt" is the reference category
"""
1=never/rarely add salt, 
2=light salt, 
3=regular salt, 
4=heavy salt, 
6=salt substitute
"""

# ALQ121: "In the past 12 months, how often did you drink alcohol?" We just remove the missing codes
data['ALQ121'] = data['ALQ121'].replace({77:np.nan, 99:np.nan})

# PAQ706: "In the past 7 days, how many days were you physically active for 60+ minutes?" Once again just remove missing codes
data['PAQ706'] = data['PAQ706'].replace({77:np.nan, 99:np.nan})

# OCQ180: "How many hours per week do you work?" Once again just remove sentinel missing codes
data['OCQ180'] = data['OCQ180'].replace({77777:np.nan, 99999:np.nan})

# SLQ300 (SLQ310): "How many hours of sleep do you usually get on weekdays (weekends)?" Remove missing codes.
data['SLQ300'] = data['SLQ300'].replace({77:np.nan, 99:np.nan})
data['SLQ310'] = data['SLQ310'].replace({77:np.nan, 99:np.nan})

# 'DIQ010': Treating 'borderline' diabetes as 0.
data['DIQ010'] = data['DIQ010'].map({1.0: 1.0, 2.0: 0.0, 3.0: 0.0, 0.0: 0.0})


In [26]:
# Column definitions followed by sanity checking of preprocessing
salt_cols = [c for c in data.columns if c.startswith('SALT_')]
race_cols  = [c for c in data.columns if c.startswith('RACE_')]

# Setting dictionaries to make it easier to idenify each variable code going forth
demo_meta = {
    'RIDAGEYR': 'Age',
    'RIAGENDR': 'Sex (0=Male, 1=Female)',
    'BMXBMI':   'BMI',
    'BMXWAIST': 'Waist circumference',
    'SMQ020':   'Ever smoked 100+ cigarettes',
    'SMQ040':   'Smoking frequency (0=none, 1=some days, 2=every day)',
    'ALQ121':   'Alcohol frequency past 12 months',
    'PAQ706':   'Days physically active past 7 days',
    'HIQ011':   'Has health insurance',
    'SLQ300':   'Weekday sleep hours',
    'SLQ310':   'Weekend sleep hours',
    'HUQ010':   'Self-rated health (1=Poor, 5=Excellent)',
    'HUQ090':   'Seen mental health professional past year',
    'OCQ180':   'Hours worked per week',
    'WHQ070':   'Tried to lose weight past year',
    'SBP':      'Systolic BP (average of 3 readings)',
    'DBP':      'Diastolic BP (average of 3 readings)',
}

disease_meta = {
    'DIQ010':  'Diabetes',
    'DIQ160':  'Pre-diabetes',
    'BPQ020':  'Hypertension',
    'BPQ080':  'High cholesterol',
    'MCQ160A': 'Arthritis',
    'MCQ160B': 'Congestive heart failure',
    'MCQ160C': 'Coronary heart disease',
    'MCQ160E': 'Heart attack',
    'MCQ160F': 'Stroke',
    'MCQ160L': 'Liver condition',
    'MCQ160M': 'Thyroid problem',
    'MCQ160P': 'COPD/emphysema',
    'MCQ220':  'Cancer',
    'KIQ022':  'Kidney disease',
    'MCQ010':  'Asthma',
}

# Derive lists from dictionaries
demo_cols    = list(demo_meta.keys()) + salt_cols + race_cols
disease_cols = list(disease_meta.keys())

# Sanity checks for disease and demographics
print("\nDisease prevalences:")
print(f"  {'Disease':<28} {'Code':<10} {'Prev %':>7}  {'N obs':>7}")
print(f"  {'-'*54}")
for col, name in disease_meta.items():
    prev = data[col].mean() * 100
    n    = data[col].notna().sum()
    print(f"  {name:<28} {col:<10} {prev:>6.1f}%  {n:>7,}")

print("\nDemo feature distributions:")
print(data[demo_cols].describe().round(2))


Disease prevalences:
  Disease                      Code        Prev %    N obs
  ------------------------------------------------------
  Diabetes                     DIQ010        9.2%   11,736
  Pre-diabetes                 DIQ160       11.5%    8,007
  Hypertension                 BPQ020       35.0%    8,487
  High cholesterol             BPQ080       36.7%    8,444
  Arthritis                    MCQ160A      32.5%    7,790
  Congestive heart failure     MCQ160B       4.4%    7,791
  Coronary heart disease       MCQ160C       5.2%    7,773
  Heart attack                 MCQ160E       4.3%    7,794
  Stroke                       MCQ160F       4.7%    7,785
  Liver condition              MCQ160L       5.5%    7,794
  Thyroid problem              MCQ160M      13.5%    7,787
  COPD/emphysema               MCQ160P       7.3%    7,793
  Cancer                       MCQ220       15.0%    7,800
  Kidney disease               KIQ022        4.1%    7,794
  Asthma                       MCQ01

In [27]:
# Redefine lists with clean col names
salt_cols = [c for c in data.columns if c.startswith('SALT_')]
race_cols = [c for c in data.columns if c.startswith('RACE_')]
demo_cols = list(demo_meta.keys()) + salt_cols + race_cols
disease_cols = list(disease_meta.keys())

# Fill NaN with -1 for lifestyle cols that are not asked to everyone
# (e.g. children not asked about smoking, alcohol, work hours, physical activity)
# -1 is a distinct value the model can learn means "not applicable" rather than zero
indicator_cols = ['SMQ040', 'ALQ121', 'PAQ706', 'OCQ180',
                  'SMQ020', 'HIQ011', 'SLQ300', 'SLQ310', 'HUQ090', 'WHQ070']
for col in indicator_cols:
    data[col] = data[col].fillna(-1)

# Drop only rows where core physical measurements are missing
# These are measured at the exam centre for everyone — if missing the row is unusable
# We keep children, we keep all ages — we just need the basic measurements to exist
core_cols = ['RIDAGEYR', 'RIAGENDR', 'BMXBMI', 'BMXWAIST', 'SBP', 'DBP']
data = data.dropna(subset=core_cols)
print(f"Rows retained: {data.shape[0]}")

# Confirm no remaining NaN in demo cols
print(f"Demo NaN remaining: {data[demo_cols].isna().sum().sum()}")

# I checked and only two values missing so just filling with -1
data[demo_cols] = data[demo_cols].fillna(-1)
print(f"Demo NaN remaining: {data[demo_cols].isna().sum().sum()}")  # should be 0

# Disease cols intentionally keep NaN — masked out per head in loss function
print("\nDisease missingness:")
print(f"  {'Disease':<28} {'Code':<10} {'Missing %':>10}  {'N obs':>7}")
print(f"  {'-'*57}")
for col, name in disease_meta.items():
    pct = data[col].isna().mean() * 100
    n   = data[col].notna().sum()
    print(f"  {name:<28} {col:<10} {pct:>9.1f}%  {n:>7,}")

Rows retained: 7269
Demo NaN remaining: 2
Demo NaN remaining: 0

Disease missingness:
  Disease                      Code        Missing %    N obs
  ---------------------------------------------------------
  Diabetes                     DIQ010           0.0%    7,268
  Pre-diabetes                 DIQ160          21.6%    5,699
  Hypertension                 BPQ020          15.3%    6,160
  High cholesterol             BPQ080          15.7%    6,131
  Arthritis                    MCQ160A         22.4%    5,640
  Congestive heart failure     MCQ160B         22.4%    5,643
  Coronary heart disease       MCQ160C         22.6%    5,628
  Heart attack                 MCQ160E         22.4%    5,640
  Stroke                       MCQ160F         22.4%    5,640
  Liver condition              MCQ160L         22.4%    5,640
  Thyroid problem              MCQ160M         22.4%    5,639
  COPD/emphysema               MCQ160P         22.4%    5,640
  Cancer                       MCQ220          2

In [28]:
# Save cleaned data
data.to_csv('saved/nhanes_clean.csv', index=False)

# Save column lists so we can reload them in the next notebook
import json
with open('saved/col_lists.json', 'w') as f:
    json.dump({
        'demo_cols':    demo_cols,
        'disease_cols': disease_cols,
        'demo_meta':    demo_meta,
        'disease_meta': disease_meta,
    }, f, indent=2)